# ARCHIVED / NON-RUNNABLE — targeted name/locality SFT v5

**QUARANTINED HISTORICAL NOTEBOOK. Do not run.** Its commands write
`v1` artifacts and do not represent the completed `v5p1` run. Use the
committed `evals/sft-v5p1/` provenance and `evals/frozen-test-v5p1/`
metrics for the actual targeted experiment.


One bounded supervised run from the frozen clean-gold-v3 LoRA adapter.
The training view contains 2,798 train rows affecting `name` or `locality`
plus 274 all-clean controls. Validation and test manifests remain pinned.

**Do not run `prepare_data.py`.** This notebook copies the pinned source
manifests, builds the audited targeted view, verifies the v3 adapter hash,
and stops at a dry-run gate before GPU training.

This is not GRPO. The run is one epoch at learning rate `5e-5` and writes
to a new Drive directory. v3 artifacts are never overwritten.

Upload `sft-targeted-name-locality-v5-pack.zip` to Drive root if the
targeted trainer changes are not yet on GitHub. The pack should contain
`scripts/train_sft.py`, `scripts/build_targeted_sft.py`,
`configs/data_targeted_name_locality.yaml`,
`configs/train_targeted_name_locality.yaml`, and their tests.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/TMFNK/local-slm-de-address-repair.git
%cd local-slm-de-address-repair
!pip install -q uv
!uv sync --extra dev

Transfer gate. If GitHub already contains the targeted SFT changes,
skip the unzip cell. Otherwise upload the pack named above to Drive root.
Do not train if the targeted builder or config is missing.


In [ ]:
!if [ -f /content/drive/MyDrive/sft-targeted-name-locality-v5-pack.zip ]; then unzip -o /content/drive/MyDrive/sft-targeted-name-locality-v5-pack.zip -d /tmp/sft-targeted-pack; cp /tmp/sft-targeted-pack/scripts/train_sft.py scripts/; cp /tmp/sft-targeted-pack/scripts/build_targeted_sft.py scripts/; cp /tmp/sft-targeted-pack/configs/data_targeted_name_locality.yaml configs/; cp /tmp/sft-targeted-pack/configs/train_targeted_name_locality.yaml configs/; cp /tmp/sft-targeted-pack/tests/test_train_sft.py tests/; cp /tmp/sft-targeted-pack/tests/test_build_targeted_sft.py tests/; fi
!test -f scripts/build_targeted_sft.py
!test -f configs/train_targeted_name_locality.yaml
!uv run pytest -q tests/test_build_targeted_sft.py tests/test_train_sft.py

Copy the raw slice and the original pinned v3 manifests from Drive.
The CSV hashes and manifest hashes are checked below. **Never rebuild
the manifests in Colab.**


In [ ]:
!mkdir -p data/raw data/manifests
!cp /content/drive/MyDrive/colab-data/dirty.csv data/raw/dirty.csv
!cp /content/drive/MyDrive/colab-data/clean.csv data/raw/clean.csv
!cp /content/drive/MyDrive/colab-data/manifests/train.json data/manifests/train.json
!cp /content/drive/MyDrive/colab-data/manifests/val.json data/manifests/val.json
!cp /content/drive/MyDrive/colab-data/manifests/test.json data/manifests/test.json
!sha256sum data/raw/dirty.csv data/raw/clean.csv

In [ ]:
!uv run python -c "
import json
from pathlib import Path
expected = {
    'train': 'f7a3defaab7aa898dbc320a398f58e317dbc6bfd7435c5c7a29f5f27f2a808cd',
    'val': 'e331cb5ac908827b00698b45aefa8f0f8f13df1d9e476f5ab7a384d21b8008bf',
    'test': '160eec15661de36d45265e03a053244311187d077b7c00e4d571d2294250350c',
}
for split, want in expected.items():
    got = json.loads(Path(f'data/manifests/{split}.json').read_text())['records_sha256']
    print(split, got)
    assert got == want, (split, got, want)
print('pinned manifests match v3')
"
!uv run python scripts/build_targeted_sft.py --config configs/data.yaml --out-dir data/manifests/targeted-name-locality-v1

Stage the frozen v3 adapter. The cell refuses a missing or changed
adapter; expected SHA-256 is the full `adapter_model.safetensors` hash.


In [ ]:
%%writefile /content/stage_v3_adapter.py
import hashlib
import json
import shutil
from pathlib import Path

src_root = Path('/content/drive/MyDrive/local-slm-de-address-repair-clean-gold-v3')
dst = Path('/content/drive/MyDrive/local-slm-de-address-repair/adapters/clean-gold-v3')
repo = Path('/content/local-slm-de-address-repair')
winner = json.loads((repo / 'evals/sft-v3/selection.json').read_text())['winner']
src = src_root / 'checkpoints' / winner
weights = src / 'adapter_model.safetensors'
expected = '7103451b9cc916920c59e5d68e8c3ada852b294be5eab55a0d1a9abdc77ec712'
assert weights.is_file(), f'missing v3 adapter weights: {weights}'
h = hashlib.sha256(weights.read_bytes()).hexdigest()
assert h == expected, f'v3 adapter hash mismatch: {h}'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('staged', winner, '->', dst)
print('adapter sha256:', h)


In [ ]:
!uv run python /content/stage_v3_adapter.py

Dry-run gate. Expected: train `3072`, train hash beginning
`78c1a680e9a854ab`, val `1000`, one epoch, about `192` steps, and no
warnings. Stop if the row count or hashes differ.


In [ ]:
%cd /content/local-slm-de-address-repair
!uv run python scripts/train_sft.py --config configs/train_targeted_name_locality.yaml --dry-run

Bounded training: one run only. Stop if validation damage, inventions,
schema validity, or contract validity regress. After a disconnect, repeat
the setup/staging cells and use `--resume`.


In [ ]:
!uv run python scripts/train_sft.py --config configs/train_targeted_name_locality.yaml
# Recovery only:
# !uv run python scripts/train_sft.py --config configs/train_targeted_name_locality.yaml --resume
# If training finished but validation scoring failed:
# !uv run python scripts/train_sft.py --config configs/train_targeted_name_locality.yaml --select-only

In [ ]:
!cat /content/drive/MyDrive/local-slm-de-address-repair/checkpoints-sft-targeted-name-locality-v1/selection.json
!uv run python -c "import json; p='/content/drive/MyDrive/local-slm-de-address-repair/checkpoints-sft-targeted-name-locality-v1/training_record.json'; r=json.load(open(p)); print({k:r[k] for k in ('winner','winner_selection_score','train_seconds','val_scoring_seconds','continued_adapter_in_place')}); print(r['gpu'])"

Merge the selected targeted adapter on top of v3 exactly once: base + v3
then + targeted winner. Do not overwrite the v3 merged model or GGUF.


In [ ]:
%%writefile /content/merge_targeted_sft.py
import json
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id = 'openbmb/MiniCPM5-1B'
rev = '87179e5c1f455ef22e6223592d2d61351b525bfc'
root = Path('/content/drive/MyDrive/local-slm-de-address-repair')
v3 = root / 'adapters/clean-gold-v3'
targeted_root = root / 'checkpoints-sft-targeted-name-locality-v1'
winner = json.loads((targeted_root / 'selection.json').read_text())['winner']
targeted = targeted_root / winner
out = root / 'merged-sft-targeted-name-locality-v1'
assert (v3 / 'adapter_model.safetensors').is_file(), v3
assert (targeted / 'adapter_model.safetensors').is_file(), targeted
tok = AutoTokenizer.from_pretrained(base_id, revision=rev)
base = AutoModelForCausalLM.from_pretrained(base_id, revision=rev, torch_dtype='auto', device_map='cpu')
with_v3 = PeftModel.from_pretrained(base, str(v3)).merge_and_unload()
full = PeftModel.from_pretrained(with_v3, str(targeted)).merge_and_unload()
full.save_pretrained(out)
tok.save_pretrained(out)
assert (out / 'model.safetensors').is_file(), 'merge saved no weights'
print('merged v3 +', winner, '->', out)


In [ ]:
!uv run python /content/merge_targeted_sft.py

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp && cd /content/llama.cpp && git checkout b31b71f3a076bfc4278daad442203a9c51c6e676
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --config Release -j2 --target llama-quantize


In [ ]:
!uv run --with gguf --with sentencepiece python /content/llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/local-slm-de-address-repair/merged-sft-targeted-name-locality-v1 --outfile /content/drive/MyDrive/local-slm-de-address-repair/sft-targeted-name-locality-v1-f16.gguf --outtype f16
!/content/llama.cpp/build/bin/llama-quantize /content/drive/MyDrive/local-slm-de-address-repair/sft-targeted-name-locality-v1-f16.gguf /content/drive/MyDrive/local-slm-de-address-repair/sft-Q4_K_M-targeted-name-locality-v1.gguf Q4_K_M
!uv run python -c "from pathlib import Path; p=Path('/content/drive/MyDrive/local-slm-de-address-repair/sft-Q4_K_M-targeted-name-locality-v1.gguf'); assert p.read_bytes()[:4] == b'GGUF'; print('GGUF OK, bytes:', p.stat().st_size)"